# Đánh giá thời gian và tài nguyên

Phụ trách: Trịnh Vỹ Triết — 24120474.

Chương này đo thời gian chạy và tài nguyên của pipeline thu thập trên **Google Colab CPU-only**. Mỗi lần chạy `src/crawler/main.py` ghi tệp `data/shards/<MSSV>/stats.json` do `ResourceProfiler` sinh ra. Tệp này gồm:

- Thời gian từng pha (đo bằng `time.perf_counter`) và tổng thời gian chạy.
- Peak RSS và Average RSS của tiến trình Python, lấy mẫu bằng `psutil` trên một luồng nền. 1 MB = 1024² byte.
- Dung lượng các thư mục `data/raw/sources`, `data/raw/unzipped` và shard trước/sau khi chạy, cùng tổng `size_before_bytes`/`size_after_bytes` của các bài đã tải.
- Môi trường chạy: phiên bản Python và thư viện, số CPU, RAM máy, có phải Colab hay không.

Ngoài ra, `/usr/bin/time -v` đo độc lập cả tiến trình để đối chiếu.

> **Nguyên tắc:** chỉ dùng số đo thật. Bản ghi có `is_mock = true` (dữ liệu sinh bởi `generate_mock_data.py`) được tách riêng và **không** đưa vào kết luận.

## Mục lục

1. Môi trường Google Colab CPU-only
2. Đo thời gian từng pha
3. Peak RAM, Average RAM và dung lượng
4. Kết quả và nhận xét

In [ ]:
import json
import os
import re
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

# Tìm thư mục gốc repo để notebook chạy được từ docs/ hoặc từ gốc.
ROOT = Path.cwd()
while not (ROOT / "pyproject.toml").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
SHARDS_DIR = ROOT / "data" / "shards"

MEMBERS = {
    "24120282": "Nguyễn Quốc Đạt",
    "24120237": "Nguyễn Đình Tuấn",
    "24120438": "Trần Nguyên Tân",
    "24120470": "Dương Phan Minh Trí",
    "24120474": "Trịnh Vỹ Triết",
}
BYTES_PER_MB = 1024**2

In [ ]:
def load_stats(member_id: str) -> dict | None:
    path = SHARDS_DIR / member_id / "stats.json"
    if not path.exists():
        return None
    with open(path, encoding="utf-8") as f:
        return json.load(f)


all_stats = {mid: load_stats(mid) for mid in MEMBERS}
missing = [mid for mid, s in all_stats.items() if s is None]
mock = [mid for mid, s in all_stats.items() if s is not None and s.get("is_mock", True)]
real = {mid: s for mid, s in all_stats.items() if s is not None and not s.get("is_mock", True)}

print(f"Chưa có stats.json: {missing or 'không'}")
print(f"Dữ liệu mẫu (bị loại khỏi phân tích): {mock or 'không'}")
print(f"Số đo thật: {list(real) or 'chưa có'}")
for mid, s in real.items():
    if s.get("status") != "completed":
        print(f"  ! {mid}: status = {s.get('status')} ({s.get('error')}) — lần chạy chưa hoàn tất")

## 1. Môi trường Google Colab CPU-only

Quy trình chạy trên Colab (Runtime → Change runtime type → **CPU**):

```bash
git clone https://github.com/<username>/arxiv-harvester.git
cd arxiv-harvester
pip install -e .
/usr/bin/time -v python -m src.crawler.main \
    --member-id 24120474 --start-month 2024-06 --start-id 12938 \
    --end-month 2024-06 --end-id 17937 --output-dir data/shards/24120474 \
    2> data/shards/24120474/time_v.txt
```

Nếu phiên Colab bị ngắt giữa chừng, chạy lại đúng lệnh trên và thêm `--resume`. Các bài đã xong được bỏ qua nhờ checkpoint trong `data/shards/24120474/_checkpoint/`.

Cell dưới đây chạy lệnh trên ngay trong notebook khi đặt `RUN_BENCHMARK = True`. Mặc định cell không chạy gì, để việc build Jupyter Book không kích hoạt thu thập dữ liệu.

In [ ]:
RUN_BENCHMARK = False  # Đặt True trên Colab để chạy thật.
MEMBER_ID = "24120474"
RUN_ARGS = [
    "--member-id", MEMBER_ID,
    "--start-month", "2024-06", "--start-id", "12938",
    "--end-month", "2024-06", "--end-id", "17937",
    "--output-dir", f"data/shards/{MEMBER_ID}",
]

if RUN_BENCHMARK:
    import subprocess

    time_log = SHARDS_DIR / MEMBER_ID / "time_v.txt"
    time_log.parent.mkdir(parents=True, exist_ok=True)
    resume = ["--resume"] if (SHARDS_DIR / MEMBER_ID / "_checkpoint").exists() else []
    with open(time_log, "w", encoding="utf-8") as err:
        subprocess.run(
            ["/usr/bin/time", "-v", "python", "-m", "src.crawler.main", *RUN_ARGS, *resume],
            cwd=ROOT, stderr=err, check=False,
        )
    all_stats[MEMBER_ID] = load_stats(MEMBER_ID)
    real[MEMBER_ID] = all_stats[MEMBER_ID]

In [ ]:
env_rows = []
for mid, s in real.items():
    env = s.get("environment", {})
    env_rows.append({
        "MSSV": mid,
        "Colab": env.get("is_google_colab"),
        "CPU-only": env.get("cpu_only"),
        "Python": env.get("python_version"),
        "CPU logic": env.get("cpu_count_logical"),
        "RAM máy (MB)": env.get("total_memory_mb"),
        "Thư viện": ", ".join(f"{k} {v}" for k, v in env.get("library_versions", {}).items()),
        "batch_size": s.get("run_args", {}).get("batch_size"),
        "rate_limit_sec": s.get("run_args", {}).get("rate_limit_sec"),
        "Resume": s.get("resumed"),
    })
env_df = pd.DataFrame(env_rows)
env_df if not env_df.empty else print("Chưa có số đo thật.")

In [ ]:
# Đọc kết quả /usr/bin/time -v (nếu có) để đối chiếu với số đo của psutil.
TIME_FIELDS = {
    "Elapsed (wall clock) time (h:mm:ss or m:ss)": "wall_clock",
    "User time (seconds)": "user_sec",
    "System time (seconds)": "system_sec",
    "Percent of CPU this job got": "cpu_percent",
    "Maximum resident set size (kbytes)": "max_rss_kb",
}


def parse_wall_clock(value: str) -> float:
    parts = [float(p) for p in value.split(":")]
    seconds = 0.0
    for part in parts:
        seconds = seconds * 60 + part
    return seconds


def parse_time_v(path: Path) -> dict | None:
    if not path.exists():
        return None
    result = {}
    for line in path.read_text(encoding="utf-8", errors="replace").splitlines():
        key, sep, value = line.strip().rpartition(": ")
        if sep and key in TIME_FIELDS:
            result[TIME_FIELDS[key]] = value.strip()
    if not result:
        return None
    return {
        "wall_clock_sec": parse_wall_clock(result["wall_clock"]) if "wall_clock" in result else None,
        "user_sec": float(result.get("user_sec", "nan")),
        "system_sec": float(result.get("system_sec", "nan")),
        "cpu_percent": result.get("cpu_percent"),
        # GNU time báo kbytes = 1024 byte.
        "max_rss_mb": int(result["max_rss_kb"]) / 1024 if "max_rss_kb" in result else None,
    }


time_v = {mid: parse_time_v(SHARDS_DIR / mid / "time_v.txt") for mid in real}
pd.DataFrame({mid: v for mid, v in time_v.items() if v}).T

## 2. Đo thời gian từng pha

Các pha được đo trong `main.py`:

| Pha | Nội dung |
| :--- | :--- |
| `id_generation` | Sinh dải ID được giao (`id_range.py`) |
| `discovery` | Truy vấn metadata và versions theo lô (`discovery.py`) |
| `download_clean` | Tải e-print, giải nén, xóa ảnh (`source_downloader.py`) |
| `references` | Lấy trích dẫn Semantic Scholar (`reference_scraper.py`) |
| `storage` | Ghi Parquet từng lô và gộp tệp cuối (`storage.py`) |

Thời gian mỗi pha là tổng cộng dồn qua mọi lô và đã bao gồm thời gian chờ do giới hạn tốc độ API (`--rate-limit-sec`). Vì vậy pha nào gọi mạng thì thường bị chi phối bởi thời gian chờ chứ không phải CPU. Với lần chạy có `--resume`, `stats.json` chỉ chứa thời gian của **lần chạy đó**; các số đếm (`results`) thì cộng dồn từ checkpoint.

In [ ]:
PHASES = ["id_generation", "discovery", "download_clean", "references", "storage"]

phase_df = pd.DataFrame(
    {mid: {p: s.get("phases", {}).get(p, {}).get("elapsed_sec", 0.0) for p in PHASES} for mid, s in real.items()}
).T
if phase_df.empty:
    print("Chưa có số đo thật.")
else:
    phase_df["tổng các pha"] = phase_df[PHASES].sum(axis=1)
    phase_df["runtime_sec"] = [real[mid]["runtime_sec"] for mid in phase_df.index]
    phase_df["bài thành công"] = [real[mid].get("results", {}).get("succeeded") for mid in phase_df.index]
    phase_df["xử lý lần này"] = [real[mid].get("results", {}).get("processed_this_run") for mid in phase_df.index]
    phase_df["giây / ID"] = phase_df["runtime_sec"] / phase_df["xử lý lần này"].where(phase_df["xử lý lần này"] > 0)
    display(phase_df.round(2))

In [ ]:
if not phase_df.empty:
    fig, ax = plt.subplots(figsize=(9, 4))
    (phase_df[PHASES] / 60).plot(kind="barh", stacked=True, ax=ax)
    ax.set_xlabel("Thời gian (phút)")
    ax.set_ylabel("MSSV")
    ax.set_title("Thời gian từng pha của pipeline (Google Colab CPU-only)")
    ax.legend(loc="lower right", fontsize=8)
    plt.tight_layout()
    plt.show()

    share = phase_df[PHASES].div(phase_df[PHASES].sum(axis=1), axis=0) * 100
    display(share.round(1).add_suffix(" (%)"))

## 3. Peak RAM, Average RAM và dung lượng

- **Peak RSS** là giá trị RSS lớn nhất trong các mẫu `psutil.Process().memory_info().rss`. **Average RSS** là trung bình các mẫu đó, lấy theo chu kỳ cố định `sample_interval_sec`.
- `os_reported_peak_rss_mb` là peak RSS do hệ điều hành ghi lại (`ru_maxrss` trên Linux). Giá trị này bắt được cả đỉnh ngắn nằm giữa hai lần lấy mẫu, nên luôn ≥ peak lấy mẫu.
- `max_rss_mb` từ `/usr/bin/time -v` đo toàn bộ tiến trình, kể cả lúc nạp thư viện trước khi profiler bắt đầu.
- Đây đều là RAM **của tiến trình**, không phải RAM đã dùng của toàn máy Colab. Peak của các pha không được cộng lại với nhau.

In [ ]:
mem_rows = []
for mid, s in real.items():
    mem = s.get("memory", {})
    tv = time_v.get(mid) or {}
    mem_rows.append({
        "MSSV": mid,
        "peak_rss_mb (psutil)": mem.get("peak_rss_mb"),
        "avg_rss_mb (psutil)": mem.get("avg_rss_mb"),
        "peak_rss_mb (OS)": mem.get("os_reported_peak_rss_mb"),
        "max_rss_mb (/usr/bin/time -v)": tv.get("max_rss_mb"),
        "số mẫu": mem.get("num_samples"),
        "chu kỳ mẫu (s)": mem.get("sample_interval_sec"),
    })
mem_df = pd.DataFrame(mem_rows).set_index("MSSV") if mem_rows else pd.DataFrame()

if mem_df.empty:
    print("Chưa có số đo thật.")
else:
    display(mem_df)
    fig, ax = plt.subplots(figsize=(8, 4))
    mem_df[["avg_rss_mb (psutil)", "peak_rss_mb (psutil)", "peak_rss_mb (OS)"]].plot(kind="bar", ax=ax)
    ax.set_ylabel("RSS (MB, 1 MB = 1024² byte)")
    ax.set_title("Memory footprint của tiến trình thu thập")
    ax.tick_params(axis="x", rotation=0)
    plt.tight_layout()
    plt.show()

In [ ]:
disk_rows = []
for mid, s in real.items():
    res = s.get("results", {})
    dirs = s.get("disk", {}).get("dirs", {})
    before = res.get("source_size_before_bytes") or 0
    after = res.get("source_size_after_bytes") or 0
    disk_rows.append({
        "MSSV": mid,
        "nguồn trước dọn (MB)": before / BYTES_PER_MB,
        "nguồn sau dọn (MB)": after / BYTES_PER_MB,
        "giảm (%)": (1 - after / before) * 100 if before else None,
        "raw/unzipped cuối (MB)": dirs.get("raw_unzipped", {}).get("after_bytes", 0) / BYTES_PER_MB,
        "shard cuối (MB)": dirs.get("shard", {}).get("after_bytes", 0) / BYTES_PER_MB,
    })
disk_df = pd.DataFrame(disk_rows).set_index("MSSV") if disk_rows else pd.DataFrame()

if disk_df.empty:
    print("Chưa có số đo thật.")
else:
    display(disk_df.round(2))
    fig, ax = plt.subplots(figsize=(8, 4))
    disk_df[["nguồn trước dọn (MB)", "nguồn sau dọn (MB)"]].plot(kind="bar", ax=ax)
    ax.set_ylabel("Dung lượng (MB)")
    ax.set_title("Dung lượng mã nguồn LaTeX trước và sau khi xóa ảnh")
    ax.tick_params(axis="x", rotation=0)
    plt.tight_layout()
    plt.show()

## 4. Kết quả và nhận xét

Bảng tổng hợp bên dưới được tính trực tiếp từ các tệp `stats.json` thật.

In [ ]:
summary_rows = []
for mid, s in real.items():
    res = s.get("results", {})
    summary_rows.append({
        "MSSV": mid,
        "Họ tên": MEMBERS[mid],
        "Trạng thái": s.get("status"),
        "Tổng ID": res.get("total_ids"),
        "Thành công": res.get("succeeded"),
        "Lỗi": res.get("failed"),
        "Không tồn tại": res.get("not_found"),
        "References": res.get("references"),
        "Runtime (giờ)": s.get("runtime_sec", 0) / 3600,
        "Peak RSS (MB)": s.get("memory", {}).get("peak_rss_mb"),
        "Avg RSS (MB)": s.get("memory", {}).get("avg_rss_mb"),
    })
summary_df = pd.DataFrame(summary_rows).set_index("MSSV") if summary_rows else pd.DataFrame()
summary_df.round(2) if not summary_df.empty else print("Chưa có số đo thật.")

**Nhận xét** *(viết sau khi có số đo thật, dựa vào các bảng ở trên)*:

- Pha chiếm nhiều thời gian nhất, và mức độ phụ thuộc vào thời gian chờ rate limit so với tốc độ mạng.
- Peak RSS có ổn định qua các lô hay tăng dần (dấu hiệu giữ dữ liệu trong bộ nhớ quá lâu).
- Tỷ lệ dung lượng giảm sau khi xóa ảnh, và dung lượng còn lại trên đĩa Colab.
- Số bài lỗi hoặc không tồn tại, nguyên nhân chính (xem `results.failed_ids` trong `stats.json`).